# Осесимметричная модель: электрод – капля – ванна

Меридиональное сечение \((r, z)\) на **одной сетке** (не полный 3D). Расходная проволока, рост и отрыв капли, полёт и удар в ванну со свободной поверхностью.

**Включено**
- VOF металл / газ: свободная поверхность ванны и капля — один интерфейс
- Подача проволоки, пинч-шейка (ток в металле), отрыв и удар
- Сила Лоренца: аналитический пинч в проволоке/капле; потенциал и \(\mathbf{J}\times\mathbf{B}\) в ванне
- Гауссов нагрев и давление дуги на поверхности ванны; анодный нагрев на капле; джоулево тепло
- Фазы (линейная \(f_L\)), Дарси в каше, Марангони + CSF

**Не включено**
- Плазма дуги и магнитное поле вне металла
- Движение горелки вдоль шва (горелка стоит)

Плотность газа завышена (\(\rho_g \sim \rho_m/50\)) для устойчивости явной схемы на Python.

Свойства по умолчанию — **Al 5182** (Zhao & DebRoy, 2001).

## Схема постановки

Ось \(z\) вверх; \(z=0\) — свободная поверхность пластины. Пластина в \(z<0\), проволока и газ в \(z>0\).

```
        r=0 │  wire feed ↓ v_feed
            │  ████  R_wire
            │  ████
            │   ██   ← hanging drop (pinch neck)
            │    ◯     flight after detachment
        z=0 │ ~~~~~~~~~~~~  free surface (VOF)
            │ ████████████  plate / weld pool
   z=-H_plt │────────────────────────── r → Lr
            φ=0 (ground)   arc heat / pressure on pool surface
```

| Символ | Смысл | Типично (quick) |
|---|---|---|
| \(r, z\) | радиус; высота от поверхности | 0…6 mm; −4…+7 mm |
| \(H_\mathrm{plate}\) | толщина пластины | 4 mm |
| \(R_\mathrm{wire}\) | радиус проволоки | 0.6 mm |
| \(I\) | ток | ~150 A |
| \(v_\mathrm{feed}\) | подача | ~3.5 cm/s |


## Уравнения

Одножидкостный осесимметричный VOF:

$$
\partial_t\alpha + \nabla\cdot(\mathbf{u}\alpha)=0,
\qquad
\nabla\cdot\mathbf{u}=0
$$

Импульс (переменная плотность, CSF, Дарси, Лоренц, давление дуги):

$$
\rho(\alpha)\,(\partial_t\mathbf{u}+\mathbf{u}\cdot\nabla\mathbf{u})
= -\nabla p + \nabla\cdot(\mu\nabla\mathbf{u})
+ \rho\mathbf{g}
+ \sigma\kappa\nabla\alpha
- A_\mathrm{mush}\mathbf{u}
+ \mathbf{J}\times\mathbf{B}
+ p_\mathrm{arc}\,\mathbf{n}|\nabla\alpha|
$$

**Пинч** в металле, связанном с проволокой (\(R_m(z)\) — радиус металла):

$$
J_z = \frac{I}{\pi R_m^2},\quad
B_\theta = \frac{\mu_0 I r}{2\pi R_m^2},\quad
F_r = -J_z B_\theta
$$

**Ванна:** \(\nabla\cdot(\sigma_e\nabla\varphi)=0\), ток входит гауссом через свободную поверхность, \(\varphi=0\) на дне.

Энергия — эффективная теплоёмкость в каше, гауссов/анодный нагрев на интерфейсе, джоулево \(j^2/\sigma_e\).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name != 'heat+flow+em_3d_axi':
    cand = ROOT / 'heat+flow+em_3d_axi'
    if cand.is_dir():
        sys.path.insert(0, str(cand))
        MODEL_DIR = cand
    else:
        sys.path.insert(0, str(ROOT))
        MODEL_DIR = ROOT
else:
    sys.path.insert(0, str(ROOT))
    MODEL_DIR = ROOT

import importlib
import weldpool_axi

importlib.reload(weldpool_axi)
from weldpool_axi import (
    Material,
    MeshLevel,
    WeldPoolAxi,
    process_for_mesh,
    record_every_for_mesh,
)

print('model dir:', MODEL_DIR)


## Материал и параметры

Выберите уровень сетки **`MeshLevel`**: `quick` / `normal` / `fine` (кнопки ниже).
Стартовая капля почти критическая — чтобы увидеть отрыв. Это демо, не паспорт дуги.


In [ ]:
try:
    import ipywidgets as w
    from IPython.display import display

    mesh_widget = w.ToggleButtons(
        options=[
            ('quick  28×56  ~10 ms', MeshLevel.quick),
            ('normal 40×72  ~15 ms', MeshLevel.normal),
            ('fine   56×96  ~20 ms', MeshLevel.fine),
        ],
        value=MeshLevel.normal,
        description='Mesh',
        style={'description_width': 'initial'},
    )
    display(mesh_widget)
    MESH = mesh_widget.value
except ImportError:
    mesh_widget = None
    MESH = MeshLevel.normal
    print('ipywidgets не установлен → MESH=normal; pip install ipywidgets')


In [ ]:
if mesh_widget is not None:
    MESH = mesh_widget.value

mat = Material()
proc = process_for_mesh(MESH, enable_flow=True, enable_em=True)
record_every = record_every_for_mesh(MESH)

model = WeldPoolAxi(material=mat, process=proc)
print(
    f'MESH={MESH.value}, grid {proc.nr}×{proc.nz}, '
    f'domain {1e3*proc.Lr:.1f}×{1e3*proc.Lz:.1f} mm, '
    f't_end={1e3*proc.t_end:.0f} ms, em_every={proc.em_every}, '
    f'I={proc.I_arc:.0f} A, V0={model.metal_volume()*1e9:.2f} mm³'
)
print(f'record_every={record_every}s')


## Начальная геометрия

In [ ]:
def mirror_field(rc, field):
    """Reflect r≥0 field to full diametral plane for plotting."""
    r_full = np.concatenate([-rc[::-1], rc])
    f_full = np.concatenate([field[::-1, :], field], axis=0)
    return r_full, f_full


r_mm = 1e3 * model.rc
z_mm = 1e3 * model.zc
RR, ZZ = np.meshgrid(r_mm, z_mm, indexing='ij')

fig, axs = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
r_f, a_f = mirror_field(r_mm, model.alpha)
_, T_f = mirror_field(r_mm, model.T)
Rf, Zf = np.meshgrid(r_f, z_mm, indexing='ij')

im0 = axs[0].pcolormesh(Rf, Zf, a_f, shading='auto', cmap='Blues', vmin=0, vmax=1)
axs[0].contour(Rf, Zf, a_f, levels=[0.5], colors='k', linewidths=0.8)
axs[0].axhline(0.0, color='C1', ls='--', lw=0.8, label='z=0 surface')
axs[0].set_aspect('equal')
axs[0].set_xlabel('r [mm]')
axs[0].set_ylabel('z [mm]')
axs[0].set_title(r'initial $\alpha$ (metal)')
axs[0].legend(loc='upper right', fontsize=8)
plt.colorbar(im0, ax=axs[0], fraction=0.046)

im1 = axs[1].pcolormesh(Rf, Zf, T_f, shading='auto', cmap='inferno')
axs[1].contour(Rf, Zf, a_f, levels=[0.5], colors='cyan', linewidths=0.8)
axs[1].set_aspect('equal')
axs[1].set_xlabel('r [mm]')
axs[1].set_title('initial T [K]')
plt.colorbar(im1, ax=axs[1], fraction=0.046)
plt.show()

print(f'R_neck0 = {1e3*model.neck_radius():.3f} mm, components={model._n_components}')

## Расчёт

In [ ]:
diag = model.run(record_every=record_every, progress=True, record_fields=True)
print(
    f'final: t={1e3*model.t:.2f} ms, Tmax={model.T.max():.0f} K, '
    f'Rneck={1e3*model.neck_radius():.3f} mm, '
    f'comp={model._n_components}, detached={model._detached}, '
    f'vol bal={model.volume_balance_ratio():.3f}'
)
print(f'frames: {len(diag.alpha_frames)}')


## Диагностика: шейка, объём, отрыв

In [ ]:
t_ms = 1e3 * np.array(diag.time)

fig, axs = plt.subplots(2, 2, figsize=(10, 6), constrained_layout=True)

axs[0, 0].plot(t_ms, 1e3 * np.array(diag.neck_radius), 'C0')
axs[0, 0].set_ylabel(r'$R_\mathrm{neck}$ [mm]')
axs[0, 0].set_xlabel('t [ms]')
axs[0, 0].set_title('Neck radius')
axs[0, 0].grid(True, alpha=0.3)

axs[0, 1].plot(t_ms, np.array(diag.n_components), 'C1')
axs[0, 1].set_ylabel('metal components')
axs[0, 1].set_xlabel('t [ms]')
axs[0, 1].set_title('Connectivity (wire / pool / flying)')
axs[0, 1].grid(True, alpha=0.3)

V0 = diag.metal_volume[0]
axs[1, 0].plot(t_ms, 1e9 * (np.array(diag.metal_volume) - V0), label=r'$\Delta V_\mathrm{metal}$')
axs[1, 0].plot(t_ms, 1e9 * np.array(diag.wire_volume), '--', label=r'$V_\mathrm{wire}$')
axs[1, 0].plot(t_ms, 1e9 * np.array(diag.pool_volume), ':', label=r'$V_\mathrm{pool}$')
axs[1, 0].set_ylabel('volume [mm³]')
axs[1, 0].set_xlabel('t [ms]')
axs[1, 0].legend(fontsize=8)
axs[1, 0].set_title('Metal volume')
axs[1, 0].grid(True, alpha=0.3)

axs[1, 1].plot(t_ms, diag.T_max, label=r'$T_\mathrm{max}$')
axs[1, 1].plot(t_ms, diag.max_speed, label=r'$|u|_\mathrm{max}$')
axs[1, 1].set_xlabel('t [ms]')
axs[1, 1].legend(fontsize=8)
axs[1, 1].set_title(r'$T_\mathrm{max}$ [K] and speed [m/s]')
axs[1, 1].grid(True, alpha=0.3)

plt.show()

det_times = [1e3 * tt for tt, d in zip(diag.time, diag.detached) if d]
if det_times:
    print(f'first detachment flagged at t ≈ {det_times[0]:.2f} ms')
else:
    print('no detachment in this run — increase I_arc / R_drop0 or t_end')

## Кадры $\alpha$ и $T$ (отражение относительно оси)

In [ ]:
n_show = min(6, len(diag.alpha_frames))
idxs = np.linspace(0, len(diag.alpha_frames) - 1, n_show, dtype=int)

fig, axs = plt.subplots(2, n_show, figsize=(2.4 * n_show, 6), constrained_layout=True)
if n_show == 1:
    axs = np.array([[axs[0]], [axs[1]]])

for col, k in enumerate(idxs):
    r_f, a_f = mirror_field(r_mm, diag.alpha_frames[k])
    _, T_f = mirror_field(r_mm, diag.T_frames[k])
    Rf, Zf = np.meshgrid(r_f, z_mm, indexing='ij')
    t_k = 1e3 * diag.time_frames[k]

    axs[0, col].pcolormesh(Rf, Zf, a_f, shading='auto', cmap='Blues', vmin=0, vmax=1)
    axs[0, col].contour(Rf, Zf, a_f, levels=[0.5], colors='k', linewidths=0.6)
    axs[0, col].set_aspect('equal')
    axs[0, col].set_title(f'α  t={t_k:.1f} ms', fontsize=9)
    if col == 0:
        axs[0, col].set_ylabel('z [mm]')

    im = axs[1, col].pcolormesh(Rf, Zf, T_f, shading='auto', cmap='inferno', vmin=300, vmax=max(1500, model.T.max()))
    axs[1, col].contour(Rf, Zf, a_f, levels=[0.5], colors='cyan', linewidths=0.6)
    axs[1, col].set_aspect('equal')
    axs[1, col].set_xlabel('r [mm]')
    if col == 0:
        axs[1, col].set_ylabel('z [mm]')

plt.colorbar(im, ax=axs[1, :].tolist(), fraction=0.02, pad=0.02, label='T [K]')
plt.show()

## Анимация по времени

Нужен прогон с `record_fields=True`. Поля отражены относительно оси. Управление — под рисунком.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

assert diag.alpha_frames, 'No field frames — re-run with record_fields=True'

t_fr = np.array(diag.time_frames)
alpha_fr = diag.alpha_frames
T_fr = diag.T_frames
spd_fr = diag.speed_frames
Fr_fr = diag.Fr_frames

r_f0, a0 = mirror_field(r_mm, alpha_fr[0])
_, T0 = mirror_field(r_mm, T_fr[0])
Rf, Zf = np.meshgrid(r_f0, z_mm, indexing='ij')

T_vmin = float(min(f.min() for f in T_fr))
T_vmax = float(max(max(f.max() for f in T_fr), mat.T_liquidus + 50.0))
spd_vmax = float(max(f.max() for f in spd_fr)) or 1e-3
Fr_abs = float(max(np.max(np.abs(f)) for f in Fr_fr)) or 1.0

fig, axs = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)

im_a = axs[0].pcolormesh(Rf, Zf, a0, shading='auto', cmap='Blues', vmin=0, vmax=1)
ct_a = axs[0].contour(Rf, Zf, a0, levels=[0.5], colors='k', linewidths=0.8)
fig.colorbar(im_a, ax=axs[0], fraction=0.046, label=r'$\alpha$')
axs[0].set_aspect('equal')
axs[0].set_xlabel('r [mm]')
axs[0].set_ylabel('z [mm]')
title_a = axs[0].set_title(f'α  t={1e3*t_fr[0]:.2f} ms')

im_T = axs[1].pcolormesh(Rf, Zf, T0, shading='auto', cmap='inferno', vmin=T_vmin, vmax=T_vmax)
ct_T = axs[1].contour(Rf, Zf, a0, levels=[0.5], colors='cyan', linewidths=0.8)
fig.colorbar(im_T, ax=axs[1], fraction=0.046, label='T [K]')
axs[1].set_aspect('equal')
axs[1].set_xlabel('r [mm]')
axs[1].set_ylabel('z [mm]')
title_T = axs[1].set_title(f'T  t={1e3*t_fr[0]:.2f} ms')


def _clear_contours(ax, keep):
    for coll in list(ax.collections):
        if coll not in keep:
            try:
                coll.remove()
            except ValueError:
                pass


def update_alpha_T(i):
    _, a_i = mirror_field(r_mm, alpha_fr[i])
    _, T_i = mirror_field(r_mm, T_fr[i])
    im_a.set_array(a_i.ravel())
    im_T.set_array(T_i.ravel())
    title_a.set_text(f'α  t={1e3*t_fr[i]:.2f} ms')
    title_T.set_text(f'T  t={1e3*t_fr[i]:.2f} ms')
    _clear_contours(axs[0], {im_a})
    _clear_contours(axs[1], {im_T})
    axs[0].contour(Rf, Zf, a_i, levels=[0.5], colors='k', linewidths=0.8)
    axs[1].contour(Rf, Zf, a_i, levels=[0.5], colors='cyan', linewidths=0.8)
    return (im_a, im_T, title_a, title_T)


anim_alpha_T = FuncAnimation(fig, update_alpha_T, frames=len(alpha_fr), interval=180, blit=False)
plt.close(fig)
HTML(anim_alpha_T.to_jshtml())

In [ ]:
# |u|, pinch Fr, and diagnostic curves with time cursor
_, spd0 = mirror_field(r_mm, spd_fr[0])
_, Fr0 = mirror_field(r_mm, Fr_fr[0])

fig2, axs2 = plt.subplots(1, 3, figsize=(12, 4.2), constrained_layout=True)

im_s = axs2[0].pcolormesh(Rf, Zf, spd0, shading='auto', cmap='viridis', vmin=0, vmax=spd_vmax)
fig2.colorbar(im_s, ax=axs2[0], fraction=0.046, label='|u| [m/s]')
axs2[0].set_aspect('equal')
axs2[0].set_xlabel('r [mm]')
axs2[0].set_ylabel('z [mm]')
title_s = axs2[0].set_title(f'|u|  t={1e3*t_fr[0]:.2f} ms')

im_F = axs2[1].pcolormesh(Rf, Zf, Fr0, shading='auto', cmap='coolwarm', vmin=-Fr_abs, vmax=Fr_abs)
fig2.colorbar(im_F, ax=axs2[1], fraction=0.046, label=r'$F_r$ [N/m³]')
axs2[1].set_aspect('equal')
axs2[1].set_xlabel('r [mm]')
axs2[1].set_ylabel('z [mm]')
title_F = axs2[1].set_title(rf'pinch $F_r$  t={1e3*t_fr[0]:.2f} ms')

t_ms_all = 1e3 * np.array(diag.time)
axs2[2].plot(t_ms_all, 1e3 * np.array(diag.neck_radius), label=r'$R_\mathrm{neck}$')
axs2[2].plot(t_ms_all, diag.n_components, label='components')
axs2[2].plot(t_ms_all, diag.max_speed, label=r'$|u|_\mathrm{max}$')
cursor = axs2[2].axvline(1e3 * t_fr[0], color='k', ls='--', lw=1)
axs2[2].set_xlabel('t [ms]')
axs2[2].legend(fontsize=8)
axs2[2].grid(True, alpha=0.3)
axs2[2].set_title('Diagnostics')


def update_flow(i):
    _, spd_i = mirror_field(r_mm, spd_fr[i])
    _, Fr_i = mirror_field(r_mm, Fr_fr[i])
    _, a_i = mirror_field(r_mm, alpha_fr[i])
    im_s.set_array(spd_i.ravel())
    im_F.set_array(Fr_i.ravel())
    title_s.set_text(f'|u|  t={1e3*t_fr[i]:.2f} ms')
    title_F.set_text(rf'pinch $F_r$  t={1e3*t_fr[i]:.2f} ms')
    cursor.set_xdata([1e3 * t_fr[i], 1e3 * t_fr[i]])
    _clear_contours(axs2[0], {im_s})
    _clear_contours(axs2[1], {im_F})
    axs2[0].contour(Rf, Zf, a_i, levels=[0.5], colors='w', linewidths=0.7)
    axs2[1].contour(Rf, Zf, a_i, levels=[0.5], colors='k', linewidths=0.7)
    return (im_s, im_F, cursor, title_s, title_F)


anim_flow = FuncAnimation(fig2, update_flow, frames=len(alpha_fr), interval=180, blit=False)
plt.close(fig2)
HTML(anim_flow.to_jshtml())

## Свободная поверхность ванны и сила пинча

In [ ]:
# Free-surface height η(r): top metal near z=0 (depression into plate is η<0)
thr = proc.alpha_metal
eta = np.full(proc.nr, np.nan)
for i in range(proc.nr):
    cols = np.where(model.alpha[i, :] >= thr)[0]
    cols = cols[model.zc[cols] <= 2e-3]
    if cols.size:
        j = cols.max()
        eta[i] = model.zc[j] + 0.5 * model.dz * (2 * model.alpha[i, j] - 1)

fig, axs = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axs[0].plot(1e3 * model.rc, 1e3 * eta, 'C0')
axs[0].axhline(0.0, color='k', ls='--', lw=0.8)
axs[0].set_xlabel('r [mm]')
axs[0].set_ylabel(r'$\eta$ [mm]')
axs[0].set_title('Pool free-surface profile (z=0 = plate top)')
axs[0].grid(True, alpha=0.3)

r_f, Fr_f = mirror_field(r_mm, model.Fr)
Rf, Zf = np.meshgrid(r_f, z_mm, indexing='ij')
_, a_f = mirror_field(r_mm, model.alpha)
im = axs[1].pcolormesh(Rf, Zf, Fr_f, shading='auto', cmap='coolwarm')
axs[1].contour(Rf, Zf, a_f, levels=[0.5], colors='k', linewidths=0.7)
axs[1].axhline(0.0, color='w', ls=':', lw=0.8)
axs[1].set_aspect('equal')
axs[1].set_xlabel('r [mm]')
axs[1].set_ylabel('z [mm]')
axs[1].set_title(r'Pinch / Lorentz $F_r$ [N/m³]')
plt.colorbar(im, ax=axs[1], fraction=0.046)
plt.show()

print(f'pool depression ≈ {1e3*model.pool_depression():.3f} mm (into plate below z=0)')


## Проверки

1. **Объём металла** растёт согласованно с подачей проволоки (`volume_balance_ratio`).
2. **До удара** число компонент ≥ 2 (проволока+капля и ванна) или 3 после отрыва.
3. **Шейка** \(R_\mathrm{neck}(t)\) сужается перед отрывом при достаточном токе.

In [ ]:
bal = model.volume_balance_ratio()
max_comp = max(diag.n_components)
min_neck = min(diag.neck_radius)
R0 = diag.neck_radius[0]

print(f'volume balance ratio = {bal:.4f}  (smaller is better)')
print(f'max metal components = {max_comp}')
print(f'neck: {1e3*R0:.3f} → {1e3*min_neck:.3f} mm')
print(f'detached flag = {model._detached}')

ok_vol = bal < 0.15
ok_comp = max_comp >= 2
ok_neck = min_neck < 0.95 * R0 + 1e-9
print('checks:', {
    'volume_vs_feed': ok_vol,
    'multi_component': ok_comp,
    'neck_shrinks': ok_neck or model._detached,
})